# Holdout novelty vs perception radius

The OOD holdout `{H_10, C_11, N_13, O_10}` was defined by clustering SOAP descriptors at **r_cut = 4.5 Bohr**, but the models build their message-passing graph at a larger cutoff (**EGNN ≈ 5.25 Bohr**, **SG-8 = 8 Bohr**). The holdout then *leaks* at the model's radius. We quantify this **without clustering**, so the result does not depend on any HDBSCAN tuning.

**Novelty metric** (per element, per radius `r`):

1. **SOAP** @ `r` → a rotation/translation/permutation-invariant vector for each atom's environment.
2. **PCA on the full dataset** → low-dim coordinates — the embedding the holdout was *selected* in. At 4.5 Bohr we keep the original component counts (H 20 / C 35 / N 30 / O 20); the other radii keep enough components for the *same retained variance*, so only `r` changes, not the information budget.
3. **Whiten by the training std** along each PC → Euclidean distance becomes Mahalanobis distance w.r.t. the training distribution.
4. **`d_k`** = distance from a held-out atom to its **15th-nearest training atom** (a coverage measure: small = training surrounds it, large = training does not reach it).
5. **Calibrate** `d_k` against a held-out split of *training* atoms — a reference set (the k-NN index) and a disjoint calibration set (the null distribution). The **novelty percentile** is the % of calibration atoms with a *smaller* `d_k`: **50 = as isolated as the median training atom** (effectively *seen*), **≈100 = training's extreme tail** (genuinely OOD).

`median_novelty_pct` per held-out cluster is the leakage metric — it falls toward 50 as the holdout leaks. Section 3 then checks whether the leaked atoms inflated the reported OOD accuracy.

In [ ]:
import glob
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

import leakage_vs_radius as lv
import sanitizing_clusters as sc

CACHE, FIGS = "leak_cache", "figs"
os.makedirs(CACHE, exist_ok=True)
os.makedirs(FIGS, exist_ok=True)

## 1. Load the atomic dataset and mark the held-out molecules

In [ ]:
CURATED = os.path.join("..", "..", "..", "data_curation", "atomic", "cluster_analysis")
tv = pd.read_pickle(os.path.join(CURATED, "train_and_val.pkl"))
te = pd.read_pickle(os.path.join(CURATED, "test.pkl"))
aimel = pd.concat([tv, te], ignore_index=True)
holdout_pos = sc.load_holdout_positions(aimel, os.path.join(CURATED, "test.pkl"))
print(f"molecules: {len(aimel)} | training: {len(tv)} | held-out: {len(holdout_pos)}")
print("held-out labels:", sorted(lv.HOLDOUT_TARGET_LABELS))

## 2. Novelty across radii

For each cutoff `r` we compute the calibrated k-NN novelty of every held-out atom vs a training reference. SOAP and PCA are cached in `leak_cache/` after the first run. The 4.5-Bohr projection uses the original H20/C35/N30/O20 components; the other radii match its per-element retained variance (constant information budget).

In [ ]:
RADII = (4.5, 5.25, 8.0)
summary, atom_novelty = lv.leakage_curve(aimel, holdout_pos, cache_dir=CACHE, radii=RADII, knn=15)
summary.to_pickle(f"{CACHE}/leakage_summary.pkl")
atom_novelty.to_pickle(f"{CACHE}/leakage_atom_novelty.pkl")

### Novelty curve for the held-out labels

One line per held-out cluster. The percentile stays high (genuinely OOD) for `H_10`/`C_11` but falls sharply for `N_13` and `O_10` as the radius approaches the SG-8 cutoff — those two leak at the model's perception radius.

In [ ]:
# median novelty percentile per held-out label across radii
label_novelty = (summary[summary.label.isin(lv.HOLDOUT_TARGET_LABELS)]
                 .pivot_table(index="label", columns="r_cut", values="median_novelty_pct"))
display(label_novelty.round(1))

fig, ax = lv.plot_leakage_curve(summary, save_path=f"{FIGS}/leakage_vs_radius.png")
plt.show()

## 3. Does the leakage inflate the reported OOD accuracy?

Split the held-out atoms into **leaked** (`novelty_pct < 50` — sits inside training density) vs **novel** at 8 Bohr, and recompute per-model scores from the cached `test_preds.pkl` — no retraining. Only `N_13` and `O_10` have leaked atoms at 8 Bohr (`H_10`/`C_11` stay fully novel), so the leaked-vs-novel split exists only for those two. Scores are repeat-mean over the 5 repeats. CCC and R² reward target spread, so Spearman (rank) and MAE (raw error) are the variance-robust reads; all four are on the same atom subsets. If `leaked` is not consistently better than `novel`, the leakage did not inflate the score.

In [ ]:
# Experiment outputs live at <repo>/experiments/atomic/{model}/fold_{k}/test_preds.pkl.
# This notebook ships in <repo>/analysis/atomic/leakage_analysis/, hence "../../..".
EXP_DIR = os.path.abspath(os.path.join(os.getcwd(), "../../../experiments/atomic"))
assert os.path.isdir(EXP_DIR), f"experiments/atomic not found at {EXP_DIR}"

MODELS = ["SG-8-12", "SG-8-5", "EGFC", "SFC2", "SGFC", "EGNN", "SGNN"]
avail = [m for m in MODELS if glob.glob(f"{EXP_DIR}/{m}/fold_*/test_preds.pkl")]
assert avail, "Table 11 needs test_preds.pkl from scripts/atomic/train_multitask.py"
strat8 = lv.stratified_ccc_by_novelty(EXP_DIR, avail, atom_novelty, r_cut=8.0, test_df=te)

for lab in ["N_13", "O_10"]:
    sub = strat8[(strat8.label4 == lab) & (strat8.stratum.isin(["leaked", "novel"]))]
    print(f"=== {lab}: leaked vs novel @ 8 Bohr (held-out atoms only) ===")
    display(sub.pivot_table(index=["model", "property"], columns="stratum",
                            values=["ccc_mean", "r2_mean", "spearman_mean", "mae_mean"]).round(4))